In [2]:
# ============================================================
# LeNet5 on FashionMNIST: Train, Evaluate, Export for Verilog
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os

# Try torchvision — if missing, download manually
try:
    import torchvision
    import torchvision.transforms as transforms
    HAVE_TORCHVISION = True
except ImportError:
    HAVE_TORCHVISION = False
    print("torchvision not installed. Will load data manually from IDX files.")


In [3]:
# FashionMNIST dataset loader
from torch.utils.data import Dataset, DataLoader
import struct

class FashionMNIST_Manual(Dataset):
    """Load FashionMNIST from pre-downloaded raw IDX files."""
    def __init__(self, root='.', train=True):
        split = 'train' if train else 't10k'
        base = os.path.join(root, 'data', 'FashionMNIST', 'raw')
        img_path = os.path.join(base, f'{split}-images-idx3-ubyte')
        lab_path = os.path.join(base, f'{split}-labels-idx1-ubyte')
        if not os.path.exists(img_path):
            # Download using torchvision if available
            try:
                import torchvision
                torchvision.datasets.FashionMNIST(root=os.path.join(root,'data'), train=train, download=True)
            except ImportError:
                raise FileNotFoundError(
                    f"FashionMNIST data not found at {img_path}.\n"
                    f"Either install torchvision (pip install torchvision) or\n"
                    f"download the dataset manually from https://github.com/zalandoresearch/fashion-mnist")
        with open(img_path, 'rb') as f:
            _, num, rows, cols = struct.unpack('>IIII', f.read(16))
            self.images = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
        with open(lab_path, 'rb') as f:
            _, num = struct.unpack('>II', f.read(8))
            self.labels = np.frombuffer(f.read(), dtype=np.uint8)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = np.pad(self.images[idx], 2, mode='constant').astype(np.float32) / 255.0
        return torch.from_numpy(img).unsqueeze(0), int(self.labels[idx])

BATCH_SIZE = 256
td = FashionMNIST_Manual(train=True)
tl = DataLoader(td, batch_size=BATCH_SIZE, shuffle=True)
vd = FashionMNIST_Manual(train=False)
vl = DataLoader(vd, batch_size=BATCH_SIZE, shuffle=False)
print(f'Dataset: {len(td)} train + {len(vd)} test images')


Dataset: 60000 train + 10000 test images


In [4]:
class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool1 = nn.AvgPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.pool2 = nn.AvgPool2d(2, 2)
        self.conv3 = nn.Conv2d(16, 120, 5)
        self.fc1   = nn.Linear(120, 84)
        self.fc2   = nn.Linear(84, 10)
        self.act   = nn.Hardtanh(-1.0, 1.0) # Matches clamp_act_FB!

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.pool1(x)
        x = self.act(self.conv2(x))
        x = self.pool2(x)
        x = self.act(self.conv3(x))
        x = torch.flatten(x, 1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = LeNet5().to(device)
print(f'Model on {device}')


Model on cpu


In [5]:
# Load pre-trained weights from lenet5.pth if available, else train 3 epochs & save
pth_path = 'lenet5.pth'
if os.path.exists(pth_path):
    model.load_state_dict(torch.load(pth_path, map_location=device))
    print(f'Successfully loaded pre-trained model weights from {pth_path}')
else:
    print('Training model for 3 epochs...')
    opt = optim.Adam(model.parameters(), 0.001)
    cri = nn.CrossEntropyLoss()
    for ep in range(3):
        model.train(); co=0; to=0
        for img, lab in tl:
            img, lab = img.to(device), lab.to(device)
            opt.zero_grad(); out = model(img); loss = cri(out, lab)
            loss.backward(); opt.step()
            _, p = torch.max(out, 1)
            to += lab.size(0); co += (p == lab).sum().item()
        print(f'Epoch {ep+1}: train acc = {100*co/to:.2f}%')
    torch.save(model.state_dict(), pth_path)
    print(f'Saved trained model weights to {pth_path}')


Successfully loaded pre-trained model weights from lenet5.pth


In [6]:
model.eval(); co=0; to=0
with torch.no_grad():
    for img, lab in vl:
        img, lab = img.to(device), lab.to(device)
        _, p = torch.max(model(img), 1)
        to += lab.size(0); co += (p == lab).sum().item()
print(f'Float32 test accuracy: {100*co/to:.2f}% ({co}/{to})')


Float32 test accuracy: 86.17% (8617/10000)


In [9]:
FB = 4  # Fractional bits parameter (Q16-FB.FB format)

from pathlib import Path
import numpy as np

def quantize_to_hex(v, fb, width=16):
    v32 = np.float32(v)
    fx = int(round(v32 * (1 << fb)))
    fx = max(-32768, min(32767, fx))
    if fx < 0:
        fx += 65536
    return f"{fx:04x}"

def export_weights(model, fb, root='weights'):
    out_dir = Path(root)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, param in model.named_parameters():
        flat = param.cpu().detach().numpy().flatten()
        fname = out_dir / (name.replace(".", "_") + ".txt")
        with open(fname, 'w', encoding='utf-8', errors='ignore') as f:
            for v in flat:
                f.write(quantize_to_hex(v, fb) + "\n")
    print(f'Q{16-fb}.{fb} weights exported to {out_dir}')

export_weights(model, FB)


Q12.4 weights exported to weights


In [8]:
import struct
from pathlib import Path
import numpy as np

def export_test_images(fb, n_test=20, root='verilog/test_data'):
    out_dir = Path(root)
    out_dir.mkdir(parents=True, exist_ok=True)
    with open('data/FashionMNIST/raw/t10k-images-idx3-ubyte', 'rb') as f:
        _, num, rows, cols = struct.unpack('>IIII', f.read(16))
        test_imgs = np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)
    with open('data/FashionMNIST/raw/t10k-labels-idx1-ubyte', 'rb') as f:
        _, num = struct.unpack('>II', f.read(8))
        test_labs = np.frombuffer(f.read(), dtype=np.uint8)
    with open(out_dir / 'test_images.hex', 'w', encoding='utf-8', errors='ignore') as f:
        for idx in range(n_test):
            img = np.pad(test_imgs[idx], 2, mode='constant')
            for p in img.flatten():
                fx = int(round((float(p) / 255.0) * (1 << fb)))
                fx = max(0, min(32767, fx))
                f.write(f"{fx & 0xFFFF:04x}\n")
    with open(out_dir / 'labels.hex', 'w', encoding='utf-8', errors='ignore') as f:
        for idx in range(n_test):
            f.write(f"{int(test_labs[idx])}\n")
    print(f'Exported {n_test} padded 32x32 test images to {out_dir}')

export_test_images(FB)


Exported 20 padded 32x32 test images to verilog\test_data
